# Signal Pop AI主播 — SadTalker

自动生成：主播照片(Pollinations) → 口型同步视频(SadTalker)

**你只需要上传TTS音频，其他自动完成。**

In [ ]:
# 1. 安装依赖
!pip install -q gdown ffmpeg-python requests pillow
!apt-get -qq install ffmpeg

In [ ]:
# 2. 克隆 SadTalker
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip install -q -r requirements.txt

In [ ]:
# 3. 下载预训练模型
!python scripts/download_models.py

## 4. 生成AI主播照片

用Pollinations生成一张独特的主播照片。

**性别设定：** 填入 `female`（女声Xiaoxiao）或 `male`（男声Yunyang）

In [ ]:
# @title 设置主播性别
GENDER = "female" # @param ["female", "male"]
SEED = 42  # 改成不同数字得到不同面孔，比如 20260724

In [ ]:
import requests, urllib.parse, hashlib
from PIL import Image
from io import BytesIO

# 根据性别生成不同prompt
if GENDER == 'female':
    prompt = ('professional Chinese female news anchor, head and shoulders portrait, '
              'natural realistic body proportions, neutral studio background, '
              'professional broadcast look, soft lighting, no text, no distortion')
else:
    prompt = ('professional Chinese male news anchor, head and shoulders portrait, '
              'natural realistic body proportions, neutral studio background, '
              'professional broadcast look, soft lighting, no text, no distortion')

q = urllib.parse.quote(prompt)
url = f'https://image.pollinations.ai/prompt/{q}?width=768&height=768&nologo=true&seed={SEED}'
print(f'生成主播照片 seed={SEED}, gender={GENDER}...')
resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=120)
img = Image.open(BytesIO(resp.content))
face_path = 'anchor.jpg'
img.save(face_path)
print(f'✅ 生成完成: {face_path} ({img.size})')
display(img)

## 5. 上传TTS音频

把Windows本地生成的 `tts.wav` 上传上来

In [ ]:
from google.colab import files
uploaded_audio = files.upload()
audio_path = list(uploaded_audio.keys())[0]
print(f'✅ 上传音频: {audio_path}')

## 6. 生成AI主播口型视频

运行时间约5-10分钟（取决于音频长度）

In [ ]:
!python inference.py \
    --driven_audio "{audio_path}" \
    --source_image "{face_path}" \
    --result_dir "./output" \
    --preprocess crop \
    --still \
    --enhancer gfpgan

## 7. 下载主播视频

生成后自动下载到本地

In [ ]:
import glob, os
output_videos = glob.glob('./output/*.mp4')
print('生成的文件：')
for v in output_videos:
    size = os.path.getsize(v) / 1024 / 1024
    print(f'  {v} ({size:.1f}MB)')

from google.colab import files
for v in output_videos:
    files.download(v)